# Minecraft Server Hosting Setup

This notebook sets up and runs a Minecraft server on Google Colab with SSH tunneling.

## Prerequisites
- Google Drive access
- SSH key file (your-unique-id.first.pem)
- Portmap.io account for SSH tunneling

**Note**: Run cells in order. One-time setup cells are marked and only need to be run once.

## One-Time Setup

Run these cells only once to set up the environment.

In [ ]:
# Clean up existing Java installations
!sudo apt purge openjdk-* -y
!sudo apt autoremove -y

In [ ]:
# Mount Google Drive
#1 click the folder icon and select "Mount Drive"

# from google.colab import drive
# drive.mount('/content/drive')


#create server directory
!mkdir -p /content/drive/MyDrive/ms
%cd /content/drive/MyDrive/ms


In [ ]:
# Download Minecraft server (version 1.21.6) or latest version
!wget -O server.jar https://piston-data.mojang.com/v1/objects/6e64dcabba3c01a7271b4fa6bd898483b794c59b/server.jar


## Regular Server Setup

Run these cells each time you want to start the server.


In [ ]:
# Install OpenJDK 21
!sudo apt update
!sudo apt install openjdk-21-jdk -y

In [ ]:
# Verify Java installation
!java -version

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Verify directory contents and set SSH key permissions
%cd /content/drive/MyDrive/ms

In [ ]:
!ls -l
!chmod 600 your-unique-id.first.pem

## Server Configuration and Execution

The following cell runs the Minecraft server with SSH tunneling.

**Connection Details**:
- External IP: `tcp://your-unique-id:<port>`
- Maps to: `localhost:25565`

**Instructions**:
- Run the cell below to start the server
- Type Minecraft commands in the input prompt
- Type 'exit' to stop the server
- Monitor [MC] for server output and [SSH] for tunnel status

In [ ]:
import subprocess
import threading
import os
import time
import shlex  # For splitting SSH command string

# Server configuration
MIN_RAM = "6G"  # -Xms (initial RAM)
MAX_RAM = "8G"  # -Xmx (maximum RAM)
SERVER_JAR = "server.jar"

# SSH reverse tunnel command as a string
SSH_COMMAND = "ssh -v -i your-key.pem -o StrictHostKeyChecking=no -N -R 61819:localhost:25565 user@your-host.portmap.io"


def cleanup_world_lock():
    """Remove any existing session.lock file"""
    lock_file = "./world/session.lock"
    if os.path.exists(lock_file):
        print("Found existing session.lock - removing it...")
        os.remove(lock_file)


def run_server():
    """Runs the Minecraft server in a subprocess"""
    cleanup_world_lock()

    command = f"java -Xms{MIN_RAM} -Xmx{MAX_RAM} -jar {SERVER_JAR} nogui"
    process = subprocess.Popen(
        command.split(),
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1
    )
    return process


def ssh_tunnel_manager():
    """Manages SSH tunnel, restarts it if it disconnects"""
    while True:
        print("Starting SSH reverse tunnel...")
        try:
            ssh_args = shlex.split(SSH_COMMAND)
            ssh_process = subprocess.Popen(
                ssh_args,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                universal_newlines=True,
                bufsize=1
            )
            print(f"SSH tunnel started (PID: {ssh_process.pid})")
            # Read and print SSH output
            for line in ssh_process.stdout:
                print(f"[SSH] {line.strip()}")
            print("SSH tunnel disconnected, retrying in 5 seconds...")
        except Exception as e:
            print("SSH tunnel failed to start:", e)

        time.sleep(5)  # Wait before retrying


def output_reader(process, prefix=""):
    """Thread that continuously reads a process's output and prints it with a prefix"""
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(f"{prefix}{output.strip()}")


def server_manager():
    """Main server control function"""
    print("Starting Minecraft server... (Type 'exit' to stop)")

    process = run_server()
    if process.poll() is not None:
        print("Server failed to start!")
        return

    # Start Minecraft server output reader thread
    threading.Thread(target=output_reader, args=(process, "[MC] "), daemon=True).start()

    # Start auto-reconnecting SSH tunnel in background thread
    threading.Thread(target=ssh_tunnel_manager, daemon=True).start()

    try:
        while True:
            if process.poll() is not None:
                print("Minecraft server has stopped unexpectedly!")
                break

            cmd = input()
            if cmd.lower() == 'exit':
                process.stdin.write("stop\n")
                process.stdin.flush()
                time.sleep(5)
                break

            if process.poll() is None:
                process.stdin.write(f"{cmd}\n")
                process.stdin.flush()
            else:
                print("Cannot send command - server is not running")
                break

    except (KeyboardInterrupt, EOFError):
        print("\nStopping Minecraft server...")
        if process.poll() is None:
            process.stdin.write("stop\n")
            process.stdin.flush()
            time.sleep(3)

    finally:
        if process.poll() is None:
            process.terminate()
        print("Server and SSH tunnel manager thread have stopped.")


if __name__ == "__main__":
    # Check if server.jar exists
    if not os.path.exists(SERVER_JAR):
        print(f"Error: {SERVER_JAR} not found!")
        print("Please download it first")
    else:
        server_manager()
